<a href="https://colab.research.google.com/github/shirin6767saleh/code-/blob/Fnew/loopfd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np

# Define the range of m values to test
m_values = range(0, 101, 2)  # m from 0 to 100, step by 2

for m in m_values:
    print("=" * 60)
    print(f"Testing for m = {m}")
    print("=" * 60)

    try:
        p = 0
        N = 4 * m + 2
        theta = 2 * p

        # Check if theta is in the valid range [0, N)
        if not (0 <= theta < N):
            print(f"Error: theta = {theta} is not in the valid range [0, {N})")
            print("Please choose a different value for p")
            continue

        print(f"Parameters: m = {m}, p = {p}, N = {N}, theta = {theta}")

        # Calculate w and G_theta
        w = np.exp((-2 * np.pi * 1j) / N)
        G_theta = np.zeros((N, N), dtype=complex)
        for k in range(N):
            for l in range(N):
                exponent = (k - theta/2) * (l - theta/2)
                G_theta[k, l] = (1/np.sqrt(N)) * (w ** exponent)

        # Identity matrix
        I = np.eye(N)

        # Calculate G_theta squared
        G_squared = np.dot(G_theta, G_theta)

        # Calculate P_theta and Q_theta
        P_theta = 0.5 * (I - G_squared)
        Q_theta = 0.5 * (I + G_squared)

        # Extract real parts
        P_theta_real = np.real(P_theta)
        Q_theta_real = np.real(Q_theta)

        # Calculate S and C
        S = np.imag(G_theta)
        C = np.real(G_theta)

        # Construct Phi matrix with safe indexing - First part
        components = []

        if p + 1 > 0:
            q_first = Q_theta_real[:p+1, :]
            components.append(q_first)

        if 2*m + 1 - p > 0:
            q_last = Q_theta_real[-(2*m+1-p):, :]
            components.append(q_last)

        if p > 0:
            p_first = P_theta_real[:p, :]
            components.append(p_first)

        if 2*m - p > 0:
            p_last = P_theta_real[-(2*m-p):, :]
            components.append(p_last)

        # Combine all rows to form Phi
        if components:
            Phi = np.vstack(components)

            # Use pseudo-inverse instead of regular inverse for stability
            Phi_inv = np.linalg.pinv(Phi)

            # Calculate the transformation: Phi @ S @ Phi_inv
            result = Phi @ S @ Phi_inv

            print(f"\nPhi shape: {Phi.shape}")
            print(f"Result shape: {result.shape}")

            # Extract the B block
            if result.shape[0] >= 2*m+2:
                B = result[2*m+2:, 2*m+2:]
                print(f"\nB matrix (shape: {B.shape}):")

                # Check if B @ (B - I) + (B - I) = 0
                I_B = np.eye(B.shape[0])
                left_side = B @ (B - I_B) + (B - I_B)

                if np.allclose(left_side, np.zeros(B.shape), atol=1e-10):
                    print("B @ (B - I) + (B - I) = 0 is TRUE")
                else:
                    print("B @ (B - I) + (B - I) = 0 is FALSE")

                # Check if B @ (B + I) - (B + I) = 0
                left_side2 = B @ (B + I_B) - (B + I_B)

                if np.allclose(left_side2, np.zeros(B.shape), atol=1e-10):
                    print("B @ (B + I) - (B + I) = 0 is TRUE")
                else:
                    print("B @ (B + I) - (B + I) = 0 is FALSE")

                # Create Vp and Vm matrices
                B_size = B.shape[0]
                Vp = np.zeros((N, N))
                Vm = np.zeros((N, N))

                # Place B + I and B - I in the bottom-left corner
                Vp[N-B_size:, :B_size] = B + I_B
                Vm[N-B_size:, :B_size] = B - I_B

            else:
                print(f"\nCannot extract block of size {2*m+2}x{2*m+2}")
                continue

        else:
            print("Error: No rows selected for Phi matrix")
            continue

        # Combine first two columns of Vp and first two columns of Vm
        M = np.hstack((Vp[:, :m], Vm[:, :m]))
        print(f"\nM matrix (shape: {M.shape}):")

        # Construct Phi matrix with safe indexing - Second part
        components2 = []

        if p + 1 > 0:
            q_first = Q_theta_real[:p+1, :]
            components2.append(q_first)

        if 2*m +1- p > 0:
            q_last = Q_theta_real[-(2*m+1-p):, :]
            components2.append(q_last)

        if p > 0:
            p_first = P_theta_real[:p, :]
            components2.append(p_first)

        if 2*m - p > 0:
            p_last = P_theta_real[-(2*m-p):, :]
            components2.append(p_last)

        # Combine all rows to form Phi
        if components2:
            Phi2 = np.vstack(components2)

            # Calculate inverse of Phi2
            Phi2_inv = np.linalg.inv(Phi2)

            # Calculate the transformation: Phi2 @ C @ Phi2_inv
            result2 = Phi2 @ C @ Phi2_inv

            if result2.shape[0] >= 2*m+2:
                A = result2[0:2*m+2, 0:2*m+2]

                # Check if A @ (A - I) + (A - I) = 0
                I_A = np.eye(A.shape[0])
                left_side = A @ (A - I_A) + (A - I_A)

                if np.allclose(left_side, np.zeros(A.shape), atol=1e-10):
                    print("A @ (A - I) + (A - I) = 0 is TRUE")
                else:
                    print("A @ (A - I) + (A - I) = 0 is FALSE")

            else:
                print(f"\nCannot extract block of size {m+1}x{m+1}")
                continue

        else:
            print("Error: No rows selected for Phi2 matrix")
            continue

        # Create Vp1 and Vm1 matrices
        A_size = A.shape[0]
        I_A = np.eye(A_size)

        Vp1 = np.zeros((N, N))
        Vp1[:A_size, :A_size] = A + I_A

        Vm1 = np.zeros((N, N))
        Vm1[:A_size, :A_size] = A - I_A

        # Combine first three columns of Vp1 and first three columns of Vm1
        N_matrix = np.hstack((Vp1[:, :m+1], Vm1[:, :m+1]))
        print(f"N matrix (shape: {N_matrix.shape}):")

        # Combine columns of M and N matrices side by side
        V = np.hstack((M, N_matrix))
        print(f"Combined matrix V (shape: {V.shape}):")

        # Calculate W = Phi⁻¹ @ V
        W = np.linalg.inv(Phi) @ V
        print(f"W matrix (shape: {W.shape}):")

        # Check if W⁻¹ @ G_theta @ W is diagonal
        transformed_matrix = np.linalg.inv(W) @ G_theta @ W
        diagonal_elements = np.diag(transformed_matrix)
        off_diagonal_elements = transformed_matrix - np.diag(diagonal_elements)

        if np.allclose(off_diagonal_elements, 0, atol=1e-10):
            print("YES - The matrix W⁻¹ @ G_theta @ W is diagonal")
        else:
            print("NO - The matrix W⁻¹ @ G_theta @ W is NOT diagonal")
            print("STOPPING: Matrix is not diagonal for m =", m)
            break  # توقف حلقه اگر ماتریس قطری نباشد

        # Calculate and print the rank of matrix W
        rank_W = np.linalg.matrix_rank(W)
        print(f"Rank of matrix W: {rank_W}")

    except Exception as e:
        print(f"Error occurred for m = {m}: {str(e)}")
        continue

    print("\n" + "=" * 60 + "\n")

print("Loop completed or stopped because matrix was not diagonal!")

Testing for m = 0
Parameters: m = 0, p = 0, N = 2, theta = 0

Phi shape: (2, 2)
Result shape: (2, 2)

B matrix (shape: (0, 0)):
B @ (B - I) + (B - I) = 0 is TRUE
B @ (B + I) - (B + I) = 0 is TRUE

M matrix (shape: (2, 0)):
A @ (A - I) + (A - I) = 0 is TRUE
N matrix (shape: (2, 2)):
Combined matrix V (shape: (2, 2)):
W matrix (shape: (2, 2)):
YES - The matrix W⁻¹ @ G_theta @ W is diagonal
Rank of matrix W: 2


Testing for m = 2
Parameters: m = 2, p = 0, N = 10, theta = 0

Phi shape: (10, 10)
Result shape: (10, 10)

B matrix (shape: (4, 4)):
B @ (B - I) + (B - I) = 0 is TRUE
B @ (B + I) - (B + I) = 0 is TRUE

M matrix (shape: (10, 4)):
A @ (A - I) + (A - I) = 0 is TRUE
N matrix (shape: (10, 6)):
Combined matrix V (shape: (10, 10)):
W matrix (shape: (10, 10)):
YES - The matrix W⁻¹ @ G_theta @ W is diagonal
Rank of matrix W: 10


Testing for m = 4
Parameters: m = 4, p = 0, N = 18, theta = 0

Phi shape: (18, 18)
Result shape: (18, 18)

B matrix (shape: (8, 8)):
B @ (B - I) + (B - I) = 0 is 